# Employee Sentiment Analysis – Final LLM Assessment

This notebook performs:

- Sentiment labeling (Positive / Negative / Neutral) with **TextBlob**
- EDA and data visualizations
- Monthly sentiment scoring
- Employee ranking
- Flight risk identification
- Linear regression model for sentiment trends

Dataset: `data/employee_feedback.csv` derived from email data.


## 1. Imports and Configuration

In [ ]:

import os
from dotenv import load_dotenv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from textblob import TextBlob
from sklearn.linear_model import LinearRegression

plt.style.use("ggplot")
sns.set(font_scale=1.1)

load_dotenv()

DATA_PATH = os.getenv("DATA_PATH", "./data/employee_feedback.csv")
OUTPUT_DIR = os.getenv("OUTPUT_DIR", "./outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Using data path:", DATA_PATH)
print("Outputs will be saved to:", OUTPUT_DIR)


## 2. Load Data

In [ ]:

# Load the dataset (CSV)
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()


### 2.1. Clean and standardize columns

In [ ]:

# Map columns to standardized names
df = df.rename(columns={
    "from": "employee_id",
    "body": "feedback_text",
    "date": "date"
})

# Keep relevant columns
df = df[["employee_id", "feedback_text", "date"]].copy()
df = df.dropna(subset=["feedback_text"])
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"])

# Add text length for EDA
df["text_length"] = df["feedback_text"].str.len()
df.head()


## 3. Exploratory Data Analysis

In [ ]:

print(df.describe(include='all'))

plt.figure(figsize=(8,4))
sns.histplot(df['text_length'], bins=30)
plt.title('Distribution of Feedback Text Length')
plt.xlabel('Characters')
plt.ylabel('Count')
plt.show()

plt.figure(figsize=(8,4))
df['date'].dt.year.value_counts().sort_index().plot(kind='bar')
plt.title('Number of Feedback Entries per Year')
plt.xlabel('Year')
plt.ylabel('Count')
plt.show()


## 4. Sentiment Analysis with TextBlob

In [ ]:

def get_polarity(text: str) -> float:
    return TextBlob(str(text)).sentiment.polarity

def label_sentiment(score: float) -> str:
    if score > 0.05:
        return "Positive"
    elif score < -0.05:
        return "Negative"
    else:
        return "Neutral"

df['sentiment_score'] = df['feedback_text'].apply(get_polarity)
df['sentiment_label'] = df['sentiment_score'].apply(label_sentiment)
df[['feedback_text', 'sentiment_score', 'sentiment_label']].head()


In [ ]:

plt.figure(figsize=(5,4))
df['sentiment_label'].value_counts().plot(kind='bar')
plt.title('Sentiment Label Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.show()

plt.figure(figsize=(8,4))
sns.histplot(df['sentiment_score'], bins=30)
plt.title('Sentiment Score Distribution')
plt.xlabel('Polarity')
plt.ylabel('Count')
plt.show()


## 5. Monthly Sentiment Scoring

In [ ]:

df['year_month'] = df['date'].dt.to_period('M').dt.to_timestamp()

monthly = (
    df.groupby('year_month')
      .agg(avg_sentiment=('sentiment_score', 'mean'),
           count=('sentiment_score', 'count'))
      .reset_index()
)
monthly.head()


In [ ]:

plt.figure(figsize=(10,4))
plt.plot(monthly['year_month'], monthly['avg_sentiment'], marker='o')
plt.title('Average Sentiment Over Time (Monthly)')
plt.xlabel('Month')
plt.ylabel('Average sentiment')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 6. Employee Ranking

In [ ]:

employee_stats = (
    df.groupby('employee_id')
      .agg(avg_sentiment=('sentiment_score', 'mean'),
           feedback_count=('sentiment_score', 'count'))
      .reset_index()
)

employee_stats['rank_positive'] = employee_stats['avg_sentiment'].rank(
    ascending=False, method='dense').astype(int)

employee_stats.sort_values('avg_sentiment', ascending=False).head(10)


## 7. Flight Risk Identification

In [ ]:

NEGATIVE_THRESHOLD = -0.1
MIN_FEEDBACKS = 3

employee_stats['flight_risk'] = np.where(
    (employee_stats['avg_sentiment'] <= NEGATIVE_THRESHOLD) &
    (employee_stats['feedback_count'] >= MIN_FEEDBACKS),
    'High',
    'Low'
)

flight_risk_employees = employee_stats[employee_stats['flight_risk'] == 'High']                         .sort_values('avg_sentiment')
flight_risk_employees.head(20)


In [ ]:

plt.figure(figsize=(6,4))
sns.scatterplot(
    data=employee_stats,
    x='feedback_count',
    y='avg_sentiment',
    hue='flight_risk'
)
plt.title('Employee Sentiment vs Feedback Count (Flight Risk)')
plt.xlabel('Feedback count')
plt.ylabel('Average sentiment')
plt.show()


## 8. Linear Regression Trend

In [ ]:

monthly_sorted = monthly.sort_values('year_month').reset_index(drop=True)
if len(monthly_sorted) > 1:
    monthly_sorted['t'] = np.arange(len(monthly_sorted))
    X = monthly_sorted[['t']]
    y = monthly_sorted['avg_sentiment']

    model = LinearRegression()
    model.fit(X, y)
    monthly_sorted['predicted_sentiment'] = model.predict(X)
    slope = model.coef_[0]
    intercept = model.intercept_
    print('Trend slope:', slope)
else:
    print('Not enough monthly data points for regression.')


In [ ]:

if 'predicted_sentiment' in monthly_sorted.columns:
    plt.figure(figsize=(10,4))
    plt.plot(monthly_sorted['year_month'], monthly_sorted['avg_sentiment'], marker='o', label='Actual')
    plt.plot(monthly_sorted['year_month'], monthly_sorted['predicted_sentiment'], linestyle='--', label='Trend')
    plt.title('Monthly Sentiment with Linear Regression Trend')
    plt.xlabel('Month')
    plt.ylabel('Average sentiment')
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.show()


## 9. Save Outputs

In [ ]:

os.makedirs(OUTPUT_DIR, exist_ok=True)
employee_stats.to_csv(os.path.join(OUTPUT_DIR, 'employee_ranking_and_flight_risk.csv'), index=False)
monthly_sorted.to_csv(os.path.join(OUTPUT_DIR, 'monthly_sentiment_trend.csv'), index=False)
df.to_csv(os.path.join(OUTPUT_DIR, 'feedback_with_sentiment.csv'), index=False)
print('Saved outputs to', OUTPUT_DIR)
